<a href="https://colab.research.google.com/github/TheGeneral-Goat/python-file-io/blob/main/Python_database_fileIO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from datetime import datetime

# used to map indices to fields for easier recognition instead of if/else loops
index_to_field = {
    0: 'Firstname',
    1: 'Lastname',
    2: 'DOB',
    3: 'Home',
    4: 'Work',
    5: 'Mobile',
    6: 'IG'
}

menu_text = """What would you like to do? Input the words in brackets.
1) Input data (Input)
2) Read data by attributes (Attributes)
3) Read data by record (Record)
4) Read all records (Read)
5) Delete record (Delete)
6) Save (Save)
7) Exit (Exit)
"""

# validates whether date of birth (DOB) is following DD-MM-YYYY format
# uses datetime.strptime to weed out invalid dates (eg 31-02)
# uses try/except to remove text / incorrect date formatting
# also check if DOB is not smaller than today and not older than 1900 (impossible)
def date_validation(DOB):
    today = datetime.today()
    try:
        datetime_DOB = datetime.strptime(DOB, "%d-%m-%Y")
        if datetime_DOB > today or datetime_DOB.year < 1900:
            return False
        return True
    except ValueError:
        return False


# allows users to input new record
def input_data(record):
    new_record = [""] * 7
    pointer = 0

    while pointer < len(index_to_field):
        value = input(f"Enter {index_to_field[pointer]}: ")

        if validate(value, pointer):
            new_record[pointer] = value
            pointer += 1
        else:
            print("Enter a new value again.")

    record.append(new_record)
    print("Successfully inputted data.")


def validate(field, i):
    # check for blank fields
    if field.strip() == "":
        print("Blank required field.")
        return False

    # check if phone number is correct, length check and format check
    elif i in [3, 4, 5]:
        if len(field) != 8 or not field.isdigit():
            print(f"Invalid {index_to_field[i]} phone number")
            return False

    # format check for DOB
    elif i == 2:
        if not date_validation(field):
            print("Invalid date of birth.")
            return False

    # length check for names
    elif i in [0, 1]:
        if len(field) > 25:
            print(f"Invalid {index_to_field[i]}.")
            return False

    # check if IG name start with @
    # and sees if it contains non-allowed character
    elif i == 6:
        if field[0] != "@" or len(field) > 25:
            print("Invalid Instagram username.")
            return False

        for n in range(1, len(field)):
            if not (field[n].isalnum() or field[n] == "_" or field[n] == "."):
                print("Invalid Instagram username.")
                return False

    return True

# saves data back into the text file for easy downloading
# akin to saving RAM data into SSD before closing program
def save_data(record):
    f = open("data.txt", "w")
    for item in record:
        f.write("|||".join(item) + "\n")
    f.close()

    print("Data successfully saved. Goodbye.")


# allows for users to read data based on their attributes / records / all records
def read_data(record, mode):
    if len(record) != 0:
        if mode == 1:
            choice_of_field = input("Which field would you like to see? Avaliable fields are Firstname, Lastname, Home, Work, Mobile: ").lower()
            p = 0
            read_index = None
            while p < len(index_to_field) and read_index == None:
                if index_to_field[p].lower() == choice_of_field:
                    read_index = p
                p += 1
            if read_index is not None:
                for count in range(len(record)):
                    print(record[count][read_index])
            else:
                print("Invalid field selection. Returning to menu.")

        elif mode == 2:
            choice_of_record = input("Which record would you like to see? Select by inputting the first name of the person. ").lower()
            found = False
            i = 0

            while i < len(record) and not found:
                if record[i][0].lower() == choice_of_record:
                    print(f"Printing record for {choice_of_record}: {record[i]}")
                    found = True
                else:
                    i += 1

        elif mode == 3:
            for i in range(len(record)):
                print(record[i])
    else:
        print("Phonebook is empty. Write data before reading.")


# allows for users to delete indivdual records
def delete_data(record):
    if len(record) == 0:
        print("Phonebook is empty.")
        return record

    target = input("Please enter the first name of person you wish to delete in the record: ").lower()
    deleted = False
    i = 0

    while i < len(record) and not deleted:
        if record[i][0].lower() == target:
            confirmation = input("Target found. Are you sure you want to delete? ").lower()
            if "yes" in confirmation:
                print(f"Deleting record with the first name {record[i][0]}")
                del record[i]
                deleted = True
        else:
            i += 1

    if deleted:
        print("Record successfully deleted.")
    else:
        print("No matching record found.")


# what happens when there is a file is read
def main():
    # flag to maintain menu without 'while true:'
    menu = True
    record = []

    if os.path.exists("data.txt"):
        textfile = open("data.txt", "r")
        i = 0

        print("data.txt exists. Now opening.")
        # read textfile into record
        record = textfile.readlines()

        while i < len(record):
            invalid_fields = [] # stores invalid fields for later editing

            # replaces new lines and field seperators with nothing
            # so we can get only the data in the field
            record[i] = record[i].replace("\n", "").strip()
            record[i] = record[i].split("|||")

            # ensures that the record[i] has the correct amount of fields (7)
            # such that program doesnt break
            if len(record[i]) != len(index_to_field):
                print(f"Record {i} has an incorrect number of fields.")
                record.pop(i)
                continue

            for j in range(len(index_to_field)):
                is_valid = validate(record[i][j], j)
                if not is_valid:
                    invalid_fields.append(j)

            if len(invalid_fields) == 0:
                print(f"Record[{i}] is all correct.")
                i += 1
            # forces user to reinput data to fit validation rules based on invalid_fields
            else:
                print(f"Record {i} has at least one error. Look at the error code above for more.")
                k = 0
                while k < len(invalid_fields):
                    corr = invalid_fields[k]
                    new_value = input(f"Input corrected data for {index_to_field[corr]}: ")
                    if validate(new_value, corr):
                        record[i][corr] = new_value
                        k += 1
                    else:
                        print("Correction invalid. Please try again.")
                print(f"-> Record {i} successfully corrected and updated in memory.\n")
                i += 1

        textfile.close()

    else:
        print("File does not exist.")
        input_for_new_file = input("Would you like to open a new file? Type yes or no. ").lower()
        if not "yes" in input_for_new_file:
            print("Goodbye.")
            menu = False

    while menu:
        print()
        choice = input(menu_text).lower().strip()

        if choice == "1" or "input" in choice:
            input_data(record)

        elif choice == "2" or "attribute" in choice:
            # Mode 1: Read by attributes
            read_data(record, 1)

        elif choice == "3" or "record" in choice:
            # Mode 2: Read single record
            read_data(record, 2)

        elif choice == "4" or "read" in choice:
            # Mode 3: Read all records
            read_data(record, 3)

        elif choice == "5" or "delete" in choice:
            delete_data(record)

        elif choice == "6" or "save" in choice:
            save_data(record)
            print("Data saved successfully.")

        elif choice == "7" or "exit" in choice:
            menu = False
            save_data(record)
            print("Goodbye!")

        else:
            print("Invalid option. Please try again.")

main()